# 🧠 AWS LLM Foundations — Hands-On Lab
### BITS Pilani | Professional AI/ML Programme

---

| # | Module | Key Concepts |
|---|--------|-------------|
| 1 | LLM Training & Artifacts | Pretraining, SFT, RLHF, checkpoints, S3 storage |
| 2 | Tool Calling | Search, Document retrieval, Code Interpreter |
| 3 | Prompt Engineering | System prompts, few-shot, CoT, constraint prompting |
| 4 | RAG Pipeline | Chunking, embeddings, FAISS, retrieval, generation |
| 5 | Generation Parameters | Tokenisation, temperature, top-p, top-k, max tokens |
| 6 | Classical ML vs LLMs | Architecture, data, training, inference deep-dive |

**AWS Services Used:** SageMaker, S3, Bedrock, OpenSearch, Secrets Manager, IAM  
**LLM:** `gemini-2.5-flash` via Google Generative AI SDK

> 💡 Run on AWS SageMaker Studio (ml.t3.medium or larger). Replace `GEMINI_API_KEY` and `S3_BUCKET` with your values before starting.


---
## ⚙️ Environment Setup
Run once. Installs all required packages.

In [ ]:
!pip install -q google-generativeai boto3 faiss-cpu sentence-transformers \
               tiktoken transformers scikit-learn pandas numpy matplotlib PyPDF2

print("✅ All packages installed successfully")


In [ ]:
import os, json, time, io, contextlib, datetime, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
warnings.filterwarnings('ignore')

import google.generativeai as genai

# ── Configuration ─────────────────────────────────────────────────────────
# Option A: Direct key (dev/notebook use)
GEMINI_API_KEY = "YOUR_GEMINI_API_KEY_HERE"   # <── replace

# Option B: AWS Secrets Manager (production pattern — uncomment below)
# import boto3, json
# client = boto3.client('secretsmanager', region_name='us-east-1')
# GEMINI_API_KEY = json.loads(
#     client.get_secret_value(SecretId='gemini-api-key')['SecretString']
# )['api_key']

genai.configure(api_key=GEMINI_API_KEY)

MODEL_NAME = "gemini-2.5-flash"
S3_BUCKET  = "your-bits-lab-bucket"   # <── replace
AWS_REGION = "us-east-1"

print(f"✅ Model   : {MODEL_NAME}")
print(f"✅ Region  : {AWS_REGION}")
print(f"✅ S3      : s3://{S3_BUCKET}")


---
# MODULE 1 — LLM Training Pipeline & Artifacts

## 1.1 The Three-Stage Training Pipeline

```
Raw Text (Internet, Books, Code, Scientific Papers)
        │
        ▼
[Stage 1] PRETRAINING          — Self-supervised next-token prediction
        │  Learns: language, facts, reasoning, world knowledge
        │  Data:   Trillions of tokens  |  Time: Weeks on 1000s of GPUs
        ▼
[Stage 2] SUPERVISED FINE-TUNING (SFT)  — Instruction-response pairs
        │  Learns: Following instructions, answering questions
        │  Data:   Thousands of curated Q&A pairs
        ▼
[Stage 3] RLHF / DPO           — Human preference feedback
        │  Learns: Helpfulness, harmlessness, honesty
        │  Data:   Ranked human comparisons
        ▼
     Final LLM  ← Gemini Flash 2.5
```

## 1.2 Training Artifacts Stored on S3

| Artifact | Description | Typical Size |
|----------|-------------|-------------|
| `model.safetensors` | Trained weight matrices | 1 GB – 700 GB |
| `config.json` | Architecture hyperparameters | ~5 KB |
| `tokenizer.json` | Vocabulary + BPE merge rules | ~3 MB |
| `training_args.json` | LR, batch size, epochs, scheduler | ~2 KB |
| `checkpoint-N/` | Full snapshot at step N | Same as model |
| `trainer_state.json` | Loss curve, eval metrics log | ~50 KB |


In [ ]:
# ── 1.3 Pretraining Objective: Next-Token Prediction ─────────────────────
# Gemini was pretrained to predict the next token given all previous tokens.
# We demonstrate this behaviour with completion prompts.

model = genai.GenerativeModel(MODEL_NAME)

pretrain_prompts = [
    "The capital of France is",
    "In machine learning, a neural network consists of",
    "Amazon S3 is a scalable object storage service used for",
    "The self-attention mechanism in Transformers allows the model to",
]

print("=" * 65)
print("PRETRAINING OBJECTIVE — Next Token / Phrase Completion")
print("=" * 65)
print("During pretraining, the model learned to fill in what comes next.")
print()

for prompt in pretrain_prompts:
    response = model.generate_content(
        f"Complete this sentence in 10 words or fewer: {prompt}",
        generation_config=genai.GenerationConfig(max_output_tokens=30, temperature=0.1)
    )
    print(f"📌 PROMPT : {prompt}")
    print(f"   OUTPUT : {response.text.strip()}")
    print()


In [ ]:
# ── 1.4 SFT Behaviour — Instruction Following ─────────────────────────────
# After SFT, the model follows structured instructions, not just completions.

sft_tasks = [
    ("Explain", "Explain gradient descent in exactly 3 bullet points."),
    ("Translate", "Translate 'Machine learning is transforming every industry' to French."),
    ("Code", "Write a Python one-liner that computes the mean of a list called `data`."),
    ("Summarise", "Summarise in 1 sentence: 'Amazon SageMaker provides a managed environment for training, tuning, and deploying ML models at scale on AWS infrastructure.'"),
]

print("=" * 65)
print("SFT BEHAVIOUR — Instruction Following (vs raw completion)")
print("=" * 65)

for task_type, instruction in sft_tasks:
    response = model.generate_content(
        instruction,
        generation_config=genai.GenerationConfig(max_output_tokens=120, temperature=0.2)
    )
    print(f"\n[{task_type}] {instruction}")
    print(f"Response: {response.text.strip()}")
    print("─" * 55)


In [ ]:
# ── 1.5 Artifact Storage — Simulating SageMaker → S3 Pipeline ────────────
# Real SageMaker Training Jobs push all artifacts to S3 automatically.

training_manifest = {
    "model_id": "gemini-2.5-flash-sft-v1",
    "base_model": "gemini-2.5-flash",
    "training_stage": "supervised_fine_tuning",
    "training_config": {
        "learning_rate": 2e-5,
        "batch_size": 32,
        "epochs": 3,
        "max_seq_length": 2048,
        "optimizer": "AdamW",
        "lr_scheduler": "cosine_with_warmup",
        "warmup_ratio": 0.05
    },
    "dataset": {
        "name": "bits_pilani_aws_qa_pairs",
        "train_samples": 48000,
        "eval_samples": 2000,
        "format": "JSONL",
        "s3_uri": f"s3://{S3_BUCKET}/datasets/train_v1.jsonl"
    },
    "checkpoints": [
        {"step": 500,  "train_loss": 2.41, "eval_loss": 2.58, "s3_uri": f"s3://{S3_BUCKET}/ckpts/step-500/"},
        {"step": 1000, "train_loss": 1.87, "eval_loss": 2.10, "s3_uri": f"s3://{S3_BUCKET}/ckpts/step-1000/"},
        {"step": 1500, "train_loss": 1.52, "eval_loss": 1.74, "s3_uri": f"s3://{S3_BUCKET}/ckpts/step-1500/"},
        {"step": 2000, "train_loss": 1.33, "eval_loss": 1.55, "s3_uri": f"s3://{S3_BUCKET}/ckpts/step-2000/"},
    ],
    "best_checkpoint": "step-2000",
    "final_model_s3": f"s3://{S3_BUCKET}/models/final/gemini-sft-v1/",
    "created_at": datetime.datetime.now().isoformat()
}

with open("/tmp/training_manifest.json", "w") as f:
    json.dump(training_manifest, f, indent=2)

print("📦 Training Artifacts Manifest (saved to /tmp/training_manifest.json):")
print("=" * 65)
print(json.dumps(training_manifest, indent=2))

# PRODUCTION: Upload to S3
# s3 = boto3.client('s3', region_name=AWS_REGION)
# s3.upload_file("/tmp/training_manifest.json", S3_BUCKET, "artifacts/training_manifest.json")
# print(f"\n✅ Uploaded → s3://{S3_BUCKET}/artifacts/training_manifest.json")


In [ ]:
# ── 1.6 Training Loss Curve Visualisation ────────────────────────────────

steps     = [100, 300, 500, 750, 1000, 1250, 1500, 1750, 2000]
train_loss = [3.8, 2.9, 2.41, 2.05, 1.87, 1.68, 1.52, 1.41, 1.33]
eval_loss  = [3.95, 3.1, 2.6, 2.28, 2.10, 1.91, 1.74, 1.63, 1.55]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

# Loss curve
ax1.plot(steps, train_loss, 'b-o', linewidth=2.5, markersize=7, label='Train Loss', zorder=3)
ax1.plot(steps, eval_loss,  'r--s', linewidth=2.5, markersize=7, label='Eval Loss', zorder=3)
ax1.fill_between(steps, train_loss, eval_loss, alpha=0.1, color='purple', label='Generalisation Gap')

for step, tl in zip([500, 1000, 1500, 2000], [2.41, 1.87, 1.52, 1.33]):
    ax1.axvline(step, color='grey', linestyle=':', linewidth=1.2, zorder=1)
    ax1.annotate(f'ckpt\n{step}', xy=(step, tl), xytext=(step+40, tl+0.18),
                 fontsize=8.5, color='steelblue',
                 arrowprops=dict(arrowstyle='->', color='steelblue', lw=1.0))

ax1.set_xlabel('Training Step', fontsize=12)
ax1.set_ylabel('Cross-Entropy Loss', fontsize=12)
ax1.set_title('LLM SFT Training — Loss Curve\n(Checkpoints → S3 every 500 steps)', fontsize=12)
ax1.legend(fontsize=10); ax1.grid(True, alpha=0.3)

# Perplexity (exp of loss)
train_ppl = [np.exp(l) for l in train_loss]
eval_ppl  = [np.exp(l) for l in eval_loss]
ax2.plot(steps, train_ppl, 'b-o', linewidth=2.5, markersize=7, label='Train PPL')
ax2.plot(steps, eval_ppl,  'r--s', linewidth=2.5, markersize=7, label='Eval PPL')
ax2.set_xlabel('Training Step', fontsize=12)
ax2.set_ylabel('Perplexity (exp(loss))', fontsize=12)
ax2.set_title('Perplexity Curve\n(Lower = Better Language Modelling)', fontsize=12)
ax2.legend(fontsize=10); ax2.grid(True, alpha=0.3)

plt.suptitle('Module 1: LLM Training Metrics', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig("/tmp/m1_training_curves.png", dpi=130, bbox_inches='tight')
plt.show()
print("📊 Perplexity = exp(loss). PPL=1 means perfect prediction; higher = more uncertain.")


---
# MODULE 2 — Tool Calling: Search · Document Retrieval · Code Interpreter

## How Tool Calling Works

```
User Query ──► LLM evaluates: "Do I need a tool?"
                    │
          ┌─────────┴──────────┐
       YES │                    │ NO
          ▼                    ▼
    Tool invoked          Direct answer
    (search/doc/code)
          │
          ▼
    Tool result returned to LLM
          │
          ▼
    LLM synthesises final grounded answer
```

**Three tools we implement:**
- `search_web(query)` — knowledge base / web search
- `get_document(doc_name)` — S3 document retrieval  
- `run_code(code)` — Python code execution and output capture


In [ ]:
# ── 2.1 Define Tool Functions ────────────────────────────────────────────

# Tool 1: Simulated Knowledge Base Search
# (Production: replace with Amazon Kendra or OpenSearch full-text search)
KNOWLEDGE_BASE = {
    "aws sagemaker":        "Amazon SageMaker is a fully managed ML service for building, training, and deploying ML models. It supports built-in algorithms, custom Docker containers, AutoML (Autopilot), Feature Store, and Model Monitor.",
    "transformer":          "Transformers use multi-head self-attention to process tokens in parallel. Components: positional encoding, Q/K/V attention heads, feed-forward layers, layer norm, residual connections.",
    "rag":                  "RAG (Retrieval-Augmented Generation) combines dense retrieval with LLM generation. Documents → chunks → embeddings → vector DB → retrieve top-k → augment prompt → generate.",
    "gemini flash":         "Gemini 2.5 Flash is Google's fastest multimodal LLM with 1M token context, native tool use, code execution, and streaming. Optimised for high-throughput, cost-effective workloads.",
    "amazon bedrock":       "Amazon Bedrock provides foundation models (Claude, Llama, Titan, Mistral) via API. Supports Knowledge Bases (RAG), Agents (multi-step), Guardrails, and fine-tuning with private data.",
    "aws lambda":           "AWS Lambda is serverless compute. Code runs on-demand, scales automatically, billed per invocation. Ideal for RAG retrieval endpoints, model inference triggers, and event-driven ML pipelines.",
}

def search_web(query: str) -> str:
    q = query.lower()
    for key, val in KNOWLEDGE_BASE.items():
        if any(kw in q for kw in key.split()):
            return f"[SEARCH RESULT — '{query}']\n{val}"
    return f"[SEARCH RESULT — '{query}']\nNo specific match. General knowledge applies."


# Tool 2: S3 Document Retrieval
SIMULATED_S3_DOCS = {
    "ml_policy.pdf": """ML GOVERNANCE POLICY v2.1 — BITS Pilani AI Lab
Section 3.2: All production ML models must have documented bias evaluations.
Section 4.1: Model cards are mandatory for customer-facing AI systems.
Section 4.3: LLM deployments require red-teaming reports before go-live.
Section 5.0: Training data must be retained for minimum 3 years.""",
    "aws_sla.txt": """AWS SERVICE LEVEL AGREEMENT — Key Terms
EC2/SageMaker Uptime SLA: 99.99% monthly uptime commitment.
Bedrock API SLA: 99.9% availability with exponential backoff recommended.
S3 Durability: 99.999999999% (11 nines) for Standard class.
Support Response: Business critical P1 issues — 15 minute response.""",
}

def get_document(doc_name: str) -> str:
    # Production: s3 = boto3.client('s3'); obj = s3.get_object(Bucket=S3_BUCKET, Key=f'docs/{doc_name}')
    if doc_name in SIMULATED_S3_DOCS:
        return f"[S3 DOC: s3://{S3_BUCKET}/docs/{doc_name}]\n{SIMULATED_S3_DOCS[doc_name]}"
    return f"[S3 DOC] '{doc_name}' not found in bucket."


# Tool 3: Code Interpreter (safe exec with captured stdout)
def run_code(code: str) -> str:
    buf = io.StringIO()
    try:
        with contextlib.redirect_stdout(buf):
            exec(code, {"__builtins__": __builtins__, "np": np, "pd": pd, "math": __import__("math")})
        out = buf.getvalue()
        return f"[CODE OUTPUT]\n{out}" if out else "[CODE OUTPUT] Executed (no stdout output)."
    except Exception as e:
        return f"[CODE ERROR] {type(e).__name__}: {e}"


TOOL_REGISTRY = {"search_web": search_web, "get_document": get_document, "run_code": run_code}
print("✅ Tool registry ready:", list(TOOL_REGISTRY.keys()))


In [ ]:
# ── 2.2 Tool-Calling Agent Loop ──────────────────────────────────────────
# The LLM decides which tool to call; we execute it; LLM synthesises the answer.

TOOL_SYSTEM_PROMPT = """You are an AWS technical assistant with access to three tools:
1. search_web(query)        — Search for technical information
2. get_document(doc_name)   — Retrieve a document (ml_policy.pdf, aws_sla.txt)
3. run_code(code)           — Execute Python code and return output

When you need a tool, respond ONLY with valid JSON (no markdown backticks):
  {"tool": "<tool_name>", "args": {"<param>": "<value>"}}

When you can answer directly, respond in plain text."""

def run_tool_agent(question: str, verbose: bool = True) -> str:
    model = genai.GenerativeModel(MODEL_NAME)
    cfg = genai.GenerationConfig(max_output_tokens=250, temperature=0.1)

    # Step 1 — LLM decides: tool or direct answer
    r1 = model.generate_content(f"{TOOL_SYSTEM_PROMPT}\n\nUser: {question}", generation_config=cfg)
    raw = r1.text.strip()

    if verbose:
        print(f"\n  🤖 LLM decision: {raw[:120]}")

    # Step 2 — Try tool execution
    tool_result = None
    try:
        clean = raw.replace("```json","").replace("```","").strip()
        call  = json.loads(clean)
        fname, fargs = call["tool"], call.get("args", {})
        if fname in TOOL_REGISTRY:
            if verbose: print(f"  🔧 Calling: {fname}({fargs})")
            tool_result = TOOL_REGISTRY[fname](**fargs)
            if verbose: print(f"  📥 Result preview: {tool_result[:160]}...")
    except (json.JSONDecodeError, KeyError):
        pass  # LLM gave direct answer

    # Step 3 — Final answer with tool context
    if tool_result:
        final_prompt = (f"{TOOL_SYSTEM_PROMPT}\n\nUser: {question}\n\n"
                        f"Tool result:\n{tool_result}\n\nAnswer the user using this context:")
        r2 = model.generate_content(final_prompt,
                                    generation_config=genai.GenerationConfig(max_output_tokens=350, temperature=0.3))
        return r2.text.strip()
    return raw

# ── Test all three tool types ────────────────────────────────────────────
tests = [
    ("SEARCH",   "What is Retrieval-Augmented Generation (RAG) and how does it work with AWS Bedrock?"),
    ("DOCUMENT", "What does our ml_policy.pdf say about LLM red-teaming requirements?"),
    ("CODE",     "Write and run Python code to calculate the compound interest on $10,000 at 8% for 5 years."),
]

for tool_type, question in tests:
    print("\n" + "="*65)
    print(f"🧪 TOOL: {tool_type}")
    print(f"❓ Q: {question}")
    answer = run_tool_agent(question)
    print(f"\n✅ Final Answer:\n{answer}")


---
# MODULE 3 — Prompt Engineering

## Prompt Anatomy

```
┌─────────────────────────────────────────────────────┐
│  SYSTEM PROMPT (developer-injected, user cannot see) │
│  — defines persona, rules, output format, guardrails │
├─────────────────────────────────────────────────────┤
│  FEW-SHOT EXAMPLES (optional)                        │
│  — shows input → expected output pairs               │
├─────────────────────────────────────────────────────┤
│  USER PROMPT (end-user input)                        │
│  — the actual question or task                       │
└─────────────────────────────────────────────────────┘
```

| Strategy | Best For | Key Technique |
|----------|----------|--------------|
| Zero-shot | Simple, clear tasks | Just ask |
| Few-shot | Format/style control | Show 2-5 examples |
| Chain-of-Thought | Reasoning, math, logic | "Think step by step" |
| Role prompting | Domain expertise | "You are a senior AWS architect..." |
| Constraint | Structured output | JSON schema in system prompt |


In [ ]:
# ── 3.1 System Prompt — Persona Injection ────────────────────────────────

def call_with_system(system: str, user: str, temp=0.4, max_tok=220) -> str:
    m = genai.GenerativeModel(model_name=MODEL_NAME, system_instruction=system)
    return m.generate_content(user,
        generation_config=genai.GenerationConfig(temperature=temp, max_output_tokens=max_tok)
    ).text.strip()

user_q = "Explain what an S3 bucket is."

personas = {
    "🏗️  AWS Architect":    "You are a senior AWS solutions architect. Give technically deep answers with best practices, IAM policies, and cost considerations.",
    "👶 Explain-to-5yo":    "You explain cloud tech to 5-year-olds using fun analogies, simple words, and short sentences. No jargon allowed.",
    "🔒 Security Auditor":  "You are a cloud security auditor. Always highlight risks, encryption requirements, access control, and compliance implications.",
    "💰 FinOps Analyst":    "You are a cloud cost optimisation analyst. Focus on pricing tiers, storage class trade-offs, and cost-saving strategies.",
}

print("SYSTEM PROMPT EFFECT — Same Question, 4 Personas")
print(f"User Question: {user_q}\n")

for persona, sys_prompt in personas.items():
    ans = call_with_system(sys_prompt, user_q, max_tok=150)
    print(f"{'─'*60}")
    print(f"{persona}")
    print(f"System: {sys_prompt[:75]}...")
    print(f"\n{ans}\n")


In [ ]:
# ── 3.2 Zero-Shot vs Few-Shot ─────────────────────────────────────────────

model = genai.GenerativeModel(MODEL_NAME)
cfg0  = genai.GenerationConfig(max_output_tokens=15, temperature=0.0)

ticket = "My EC2 instance keeps crashing after exactly 2 hours and I'm losing all unsaved work!"

zero_shot = f"""Classify this AWS support ticket as: COMPUTE | STORAGE | SERVERLESS | DATABASE | NETWORK
Return ONLY the category.
Ticket: {ticket}"""

few_shot = f"""Classify AWS tickets. Return ONLY: COMPUTE | STORAGE | SERVERLESS | DATABASE | NETWORK

Examples:
Ticket: "S3 bucket ACL is blocking cross-account access" → STORAGE
Ticket: "Lambda cold starts are adding 3 seconds latency" → SERVERLESS
Ticket: "RDS Aurora read replica lag keeps growing" → DATABASE
Ticket: "VPC peering routes not propagating" → NETWORK
Ticket: "EC2 instance won't start after AMI update" → COMPUTE

Ticket: {ticket} →"""

z = model.generate_content(zero_shot, generation_config=cfg0).text.strip()
f = model.generate_content(few_shot,  generation_config=cfg0).text.strip()

print("ZERO-SHOT vs FEW-SHOT")
print(f"Ticket: {ticket}")
print(f"\n📌 Zero-Shot  : {z}")
print(f"📌 Few-Shot   : {f}")
print("\nFew-shot guides format AND improves accuracy for ambiguous inputs.")


In [ ]:
# ── 3.3 Chain-of-Thought (CoT) Prompting ─────────────────────────────────

model = genai.GenerativeModel(MODEL_NAME)
cfg   = genai.GenerationConfig(max_output_tokens=350, temperature=0.1)

problem = """An ML team runs the following on AWS:
• SageMaker training: ml.p3.2xlarge at $3.06/hr, 4.5 hrs, run 3 times
• SageMaker endpoint: ml.m5.xlarge at $0.23/hr, 24/7 for 30 days
• S3 storage: 120 GB at $0.023/GB/month
• Data transfer out: 50 GB at $0.09/GB
What is the total monthly AWS cost?"""

direct = model.generate_content(
    f"Answer in one line with just the dollar total: {problem}", generation_config=cfg
).text.strip()

cot = model.generate_content(f"""Solve step-by-step showing each calculation:

{problem}

Step 1: Training job cost
Step 2: Endpoint cost (720 hours/month)
Step 3: S3 storage cost
Step 4: Data transfer cost
Step 5: Grand total""", generation_config=cfg).text.strip()

print("CHAIN-OF-THOUGHT PROMPTING")
print(f"Problem: {problem.strip()}")
print(f"\n❌ Direct (no CoT):\n{direct}")
print(f"\n✅ With CoT (step-by-step):\n{cot}")


In [ ]:
# ── 3.4 Constraint Prompting — Structured JSON Output ────────────────────

system_json = """You are an AWS cost analyser. 
Respond ONLY with valid JSON. No explanation, no markdown, no backticks.
Required schema: 
{
  "service": string,
  "monthly_cost_usd": number,
  "primary_cost_driver": string,
  "cost_tier": "low|medium|high|very_high",
  "top_3_recommendations": [string, string, string]
}"""

m_json = genai.GenerativeModel(model_name=MODEL_NAME, system_instruction=system_json)
resp   = m_json.generate_content(
    "Analyse cost of running 10 SageMaker ml.m5.xlarge inference endpoints 24/7 for one month.",
    generation_config=genai.GenerationConfig(max_output_tokens=300, temperature=0.0)
).text.strip()

print("CONSTRAINT PROMPTING — Enforced JSON Schema")
print(f"Raw output:\n{resp}\n")

try:
    parsed = json.loads(resp.replace("```json","").replace("```","").strip())
    print("✅ Parsed JSON:")
    for k, v in parsed.items():
        print(f"   {k}: {v}")
except json.JSONDecodeError as e:
    print(f"❌ Parse error: {e}")
print("\n💡 Production tip: Use Pydantic or json.loads() with retry on parse failure.")


---
# MODULE 4 — RAG (Retrieval-Augmented Generation) Pipeline

## Full Architecture

```
INDEXING PHASE (offline — run once or on document update)
──────────────────────────────────────────────────────────
  PDF / TXT / HTML Documents
       │
  [1] CHUNKING       → split into overlapping passages (e.g. 200 words, 40 overlap)
       │
  [2] EMBEDDING      → each chunk → dense vector via embedding model
       │              (sentence-transformers all-MiniLM-L6-v2  = 384 dims)
  [3] VECTOR STORE   → index in FAISS (local) or OpenSearch Serverless (AWS prod)

RETRIEVAL + GENERATION PHASE (online — per user query)
──────────────────────────────────────────────────────────
  User Query
       │
  [4] EMBED QUERY    → same embedding model → query vector
       │
  [5] VECTOR SEARCH  → cosine similarity → top-k most relevant chunks
       │
  [6] AUGMENT PROMPT → system prompt + retrieved context + user question
       │
  [7] GENERATE       → Gemini Flash 2.5 → grounded, cited answer
```

**AWS Production Stack:**  
Documents → S3 → Lambda (chunking/embedding) → OpenSearch Serverless → Bedrock (LLM) → API Gateway


In [ ]:
# ── 4.1 Document Corpus (AWS Knowledge Base) ─────────────────────────────

AWS_CORPUS = [
    ("SageMaker",    "Amazon SageMaker is a fully managed ML service. It provides Jupyter notebooks, built-in algorithms optimised for massive datasets, automatic model tuning (HPO), and one-click deployment to managed endpoints. SageMaker Pipelines enables CI/CD for ML workflows. Feature Store provides a centralised repository for ML features reusable across teams."),
    ("S3",           "Amazon S3 is an object storage service with 99.999999999% (11 nines) durability. Objects are stored in buckets as key-value pairs. Storage classes include Standard, Intelligent-Tiering, Standard-IA, Glacier Instant Retrieval, Glacier Flexible Retrieval, and Glacier Deep Archive. S3 supports versioning, lifecycle policies, replication, and event notifications."),
    ("Lambda",       "AWS Lambda is serverless compute that runs code without managing servers. Supports Python, Node.js, Java, Go, Ruby, .NET, and custom runtimes. Scales from 0 to thousands of concurrent executions automatically. Maximum execution time is 15 minutes. Memory configurable from 128 MB to 10 GB. Priced per invocation and GB-second of compute."),
    ("Bedrock",      "Amazon Bedrock is a fully managed service for accessing foundation models from Anthropic (Claude), Meta (Llama), Mistral, Stability AI, and Amazon (Titan) via a unified API. Supports Knowledge Bases for RAG using OpenSearch Serverless, Agents for multi-step task automation, Guardrails for content filtering, and fine-tuning with private data."),
    ("OpenSearch",   "Amazon OpenSearch Service is managed search and analytics. Supports full-text search, log analytics, and k-NN vector search for semantic similarity. OpenSearch Serverless auto-scales with no capacity planning. Used as the vector store in AWS Bedrock Knowledge Bases for production RAG pipelines. Supports ANN algorithms: HNSW, IVF, Faiss engine."),
    ("IAM",          "AWS IAM controls authentication and authorisation. Key entities: Users (humans), Groups (collections of users), Roles (assumed by services/apps), Policies (JSON permission documents). Best practices: least privilege, MFA for console access, rotate credentials, use Roles for EC2/Lambda (never embed access keys in code), enable CloudTrail for audit."),
    ("EC2",          "Amazon EC2 provides resizable compute. Instance families: t3/m6 (general purpose), c6 (compute optimised), r6 (memory optimised), p3/p4/g5 (GPU for ML training/inference), inf2 (AWS Inferentia for LLM inference). Pricing models: On-Demand, Reserved (1-3yr, up to 72% savings), Spot (up to 90% savings, interruptible), Savings Plans."),
    ("Glue",         "AWS Glue is a serverless ETL service. Data Catalog stores metadata. Glue Jobs run PySpark or Python shell scripts. Glue Crawlers auto-discover schema from S3, RDS, Redshift. Glue DataBrew provides visual data preparation. Commonly used in ML pipelines to preprocess raw data from S3 before SageMaker training."),
]

print(f"✅ Corpus: {len(AWS_CORPUS)} AWS service documents loaded")
for name, doc in AWS_CORPUS:
    print(f"   [{name}] {doc[:70]}...")


In [ ]:
# ── 4.2 Chunking Strategy ─────────────────────────────────────────────────

def chunk_document(text: str, chunk_words: int = 80, overlap_words: int = 20) -> list:
    """Split text into overlapping word-level chunks.
    
    Args:
        chunk_words:   Target words per chunk
        overlap_words: Words shared between consecutive chunks (context continuity)
    """
    words  = text.split()
    chunks, start = [], 0
    while start < len(words):
        end = min(start + chunk_words, len(words))
        chunks.append(" ".join(words[start:end]))
        if end == len(words): break
        start += chunk_words - overlap_words
    return chunks

# Build chunk store
all_chunks = []
for service_name, doc_text in AWS_CORPUS:
    for i, chunk in enumerate(chunk_document(doc_text)):
        all_chunks.append({
            "id":      f"{service_name}-chunk{i}",
            "service": service_name,
            "text":    chunk,
            "words":   len(chunk.split())
        })

df_chunks = pd.DataFrame(all_chunks)
print(f"✅ Total chunks: {len(all_chunks)}")
print(f"   Avg chunk size: {df_chunks.words.mean():.1f} words")
print(f"   Min/Max: {df_chunks.words.min()} / {df_chunks.words.max()} words")
print()
print(df_chunks[['id','service','words']].head(10).to_string(index=False))


In [ ]:
# ── 4.3 Embedding — Text → Dense Vectors ─────────────────────────────────
# sentence-transformers runs locally (no API cost for embedding)
# AWS production: Amazon Bedrock Titan Embeddings V2 or SageMaker embedding endpoint

from sentence_transformers import SentenceTransformer

print("⏳ Loading embedding model (all-MiniLM-L6-v2, 384-dim)...")
embedder   = SentenceTransformer('all-MiniLM-L6-v2')
print("✅ Embedding model loaded")

chunk_texts = [c["text"] for c in all_chunks]
embeddings  = embedder.encode(chunk_texts, show_progress_bar=True, batch_size=16)
embeddings  = embeddings.astype('float32')

print(f"\n✅ Embeddings shape: {embeddings.shape}")
print(f"   {len(all_chunks)} chunks × {embeddings.shape[1]} dimensions")

# Similarity between same vs different service chunks
from sklearn.metrics.pairwise import cosine_similarity

# SageMaker chunk vs other SageMaker chunk
sm_indices = [i for i,c in enumerate(all_chunks) if c['service']=='SageMaker']
s3_indices = [i for i,c in enumerate(all_chunks) if c['service']=='S3']

if len(sm_indices) >= 2:
    sim_same = cosine_similarity([embeddings[sm_indices[0]]], [embeddings[sm_indices[1]]])[0][0]
    sim_diff = cosine_similarity([embeddings[sm_indices[0]]], [embeddings[s3_indices[0]]])[0][0]
    print(f"\n📐 Similarity: SageMaker↔SageMaker = {sim_same:.4f}  (same topic)")
    print(f"   Similarity: SageMaker↔S3          = {sim_diff:.4f}  (different topic)")
    print("   Higher cosine similarity = more semantically related content")


In [ ]:
# ── 4.4 FAISS Vector Index ────────────────────────────────────────────────
# FAISS = Facebook AI Similarity Search — microsecond ANN lookup
# AWS production equivalent: Amazon OpenSearch Serverless k-NN index (HNSW)

import faiss

dim       = embeddings.shape[1]   # 384
emb_norm  = embeddings.copy()
faiss.normalize_L2(emb_norm)      # normalise for cosine similarity via inner product

index = faiss.IndexFlatIP(dim)    # Inner Product → cosine similarity after L2-norm
index.add(emb_norm)

print("✅ FAISS vector index built")
print(f"   Vectors indexed : {index.ntotal}")
print(f"   Dimensions      : {dim}")
print(f"   Index type      : IndexFlatIP (exact search)")
print()
print("AWS production mapping:")
print("   FAISS IndexFlatIP   →  OpenSearch k-NN (engine: faiss, space: cosinesimil)")
print("   FAISS IndexIVFFlat  →  OpenSearch k-NN (engine: faiss, method: ivf)")
print("   FAISS IndexHNSWFlat →  OpenSearch k-NN (engine: nmslib, method: hnsw)")

# Production persistence
# faiss.write_index(index, "/tmp/aws_knowledge.faiss")
# s3.upload_file("/tmp/aws_knowledge.faiss", S3_BUCKET, "vector-store/aws_knowledge.faiss")


In [ ]:
# ── 4.5 Retrieval Function ───────────────────────────────────────────────

def retrieve(query: str, top_k: int = 3, verbose: bool = True) -> list:
    """Embed query → cosine search → return top-k chunks with scores."""
    q_vec = embedder.encode([query]).astype('float32')
    faiss.normalize_L2(q_vec)
    scores, indices = index.search(q_vec, top_k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        c = all_chunks[idx].copy()
        c["score"] = round(float(score), 4)
        results.append(c)

    if verbose:
        print(f"\n🔍 Query: '{query}'")
        for r in results:
            print(f"   [{r['service']}] score={r['score']:.4f} | {r['text'][:90]}...")
    return results

# Test retrieval quality
test_queries = [
    "How do I store training data and model artifacts on AWS?",
    "What are the GPU instance types for ML training?",
    "How does Bedrock support RAG with vector search?",
    "Best practices for IAM security in ML pipelines?",
]

for q in test_queries:
    retrieve(q, top_k=2)


In [ ]:
# ── 4.6 Full RAG Generation ───────────────────────────────────────────────

def rag_answer(question: str, top_k: int = 3) -> dict:
    """
    Complete RAG pipeline:
    1. Embed query
    2. Retrieve top-k chunks from FAISS
    3. Augment prompt with retrieved context
    4. Generate grounded answer with Gemini Flash 2.5
    """
    # Retrieve
    chunks  = retrieve(question, top_k=top_k, verbose=False)
    context = "\n\n".join([f"[Source {i+1} — {c['service']}]\n{c['text']}"
                             for i, c in enumerate(chunks)])

    # Augmented prompt
    prompt = f"""You are an AWS technical expert. Answer ONLY using the context below.
If the answer is not in the context, say "That information is not in the provided documents."
Always cite which Source(s) you used.

CONTEXT:
{context}

QUESTION: {question}

ANSWER:"""

    model = genai.GenerativeModel(MODEL_NAME)
    resp  = model.generate_content(prompt,
        generation_config=genai.GenerationConfig(max_output_tokens=350, temperature=0.15))

    return {
        "question":  question,
        "sources":   [(c['service'], c['score']) for c in chunks],
        "answer":    resp.text.strip()
    }

questions = [
    "What S3 storage classes are available and when should I use each?",
    "How does Bedrock Agents differ from simple LLM API calls?",
    "What Lambda memory and timeout limits exist?",
    "Which ANN algorithms does OpenSearch support for vector search?",
]

for q in questions:
    result = rag_answer(q)
    print("\n" + "="*65)
    print(f"❓ {result['question']}")
    print(f"📦 Sources retrieved: {result['sources']}")
    print(f"\n✅ Answer:\n{result['answer']}")


In [ ]:
# ── 4.7 RAG vs No-RAG — Hallucination Comparison ────────────────────────

model = genai.GenerativeModel(MODEL_NAME)
cfg   = genai.GenerationConfig(max_output_tokens=180, temperature=0.4)

# A specific question where factual grounding matters
question = "What are the exact ANN algorithm options available in Amazon OpenSearch for k-NN vector search?"

no_rag  = model.generate_content(question, generation_config=cfg).text.strip()
rag_res = rag_answer(question)

print("RAG vs NO-RAG — Hallucination Risk")
print(f"\nQuestion: {question}")
print(f"\n❌ WITHOUT RAG (LLM memory only — may be outdated/wrong):")
print(no_rag)
print(f"\n✅ WITH RAG (grounded in retrieved AWS documentation):")
print(rag_res['answer'])
print(f"\n📌 Retrieved from: {rag_res['sources']}")
print("\n💡 RAG constrains the model to provided context, reducing confabulation.")


---
# MODULE 5 — Tokenisation & Generation Parameters

## The Token Pipeline

```
Input text
    │
[1] TOKENISE         → text → token IDs  (vocabulary ~32K–128K tokens)
    │                   "un" + "believ" + "able" → [1024, 8291, 629]
    │
[2] EMBEDDING LOOKUP → token IDs → dense vectors (model input)
    │
[3] TRANSFORMER FORWARD PASS
    │
[4] LOGIT OUTPUT     → one float per vocabulary token
    │
[5] ÷ TEMPERATURE    → sharpen (T<1) or flatten (T>1) the distribution
    │
[6] SOFTMAX          → logits → probabilities summing to 1.0
    │
[7] TOP-K FILTER     → keep only top K most probable tokens
    │
[8] TOP-P FILTER     → keep tokens until cumulative prob ≥ P
    │
[9] SAMPLE           → draw one token from remaining distribution
    │
    └──→ repeat until EOS token OR max_tokens reached
```


In [ ]:
# ── 5.1 Tokenisation ────────────────────────────────────────────────────
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")  # GPT-4 / reference tokeniser

examples = [
    "Hello, world!",
    "Amazon SageMaker training job",
    "tokenisation breaks text into sub-word units",
    "AWS_REGION = 'us-east-1'  # Python config",
    "The transformer architecture uses self-attention mechanisms",
    "LLMs are trained via next-token prediction on trillions of tokens",
    "🤖 AI/ML emoji and Unicode handling",
]

print("TOKENISATION — Text → Token IDs → Sub-word Units")
print("="*65)

for text in examples:
    ids    = enc.encode(text)
    tokens = [enc.decode([i]) for i in ids]
    print(f"\nText    : {text}")
    print(f"Tokens  : {tokens}")
    print(f"IDs     : {ids}")
    print(f"Count   : {len(ids)} tokens")

# Cost estimation
print("\n" + "="*65)
print("TOKEN COUNT → COST ESTIMATION (Gemini Flash 2.5 pricing)")
doc = " ".join(["This is a sample AWS ML documentation sentence."] * 200)
n_in = len(enc.encode(doc))
n_out = 500  # assumed output
cost_in  = (n_in  / 1_000_000) * 0.075   # $0.075/M input tokens
cost_out = (n_out / 1_000_000) * 0.30    # $0.30/M output tokens
print(f"  Input tokens   : {n_in:,}")
print(f"  Output tokens  : {n_out}")
print(f"  Input cost     : ${cost_in:.5f}")
print(f"  Output cost    : ${cost_out:.5f}")
print(f"  Total per call : ${cost_in+cost_out:.5f}")
print(f"  Cost for 1M calls: ${(cost_in+cost_out)*1_000_000:.2f}")


In [ ]:
# ── 5.2 Temperature — Creativity vs Determinism ──────────────────────────

model  = genai.GenerativeModel(MODEL_NAME)
prompt = "Complete this sentence: The most important thing to know about deploying LLMs on AWS is"

temps = [0.0, 0.5, 1.0, 1.8]
RUNS  = 3

print("TEMPERATURE EFFECT — Same Prompt, Different Randomness")
print(f"Prompt: {prompt}\n")

for temp in temps:
    responses = []
    for _ in range(RUNS):
        r = model.generate_content(prompt,
            generation_config=genai.GenerationConfig(temperature=temp, max_output_tokens=40)
        ).text.strip()
        responses.append(r)

    unique = len(set(responses))
    desc   = {0.0: "Fully deterministic", 0.5: "Focused + slight variety",
               1.0: "Balanced", 1.8: "Highly creative/unpredictable"}[temp]
    
    print(f"🌡️  T={temp} — {desc} | Unique/3 runs: {unique}/3")
    for i, r in enumerate(responses, 1):
        print(f"   Run {i}: {r[:100]}")
    print()


In [ ]:
# ── 5.3 Top-P, Top-K, Max Tokens — Detailed Comparison ──────────────────

model  = genai.GenerativeModel(MODEL_NAME)
prompt = "List important AWS services for building a production ML pipeline:"

configs = [
    {"label": "Greedy (T=0, K=1)",              "temperature": 0.0,  "top_p": 1.0,  "top_k": 1,   "max_output_tokens": 120},
    {"label": "Narrow nucleus (T=0.7, P=0.5)",  "temperature": 0.7,  "top_p": 0.5,  "top_k": 50,  "max_output_tokens": 120},
    {"label": "Standard (T=0.7, P=0.9, K=40)",  "temperature": 0.7,  "top_p": 0.9,  "top_k": 40,  "max_output_tokens": 120},
    {"label": "Wide vocab (T=0.9, P=0.99)",      "temperature": 0.9,  "top_p": 0.99, "top_k": 100, "max_output_tokens": 120},
    {"label": "Short output (max=25 tokens)",    "temperature": 0.5,  "top_p": 0.9,  "top_k": 40,  "max_output_tokens": 25},
    {"label": "Long output (max=300 tokens)",    "temperature": 0.5,  "top_p": 0.9,  "top_k": 40,  "max_output_tokens": 300},
]

print("TOP-P / TOP-K / MAX_TOKENS — Parameter Effects")
print(f"Prompt: {prompt}\n")

for cfg_d in configs:
    label = cfg_d.pop("label")
    resp  = model.generate_content(prompt,
        generation_config=genai.GenerationConfig(**cfg_d)
    ).text.strip()
    cfg_d["label"] = label
    n_tok = len(enc.encode(resp))
    print(f"⚙️  {label}")
    print(f"   Params: T={cfg_d['temperature']} P={cfg_d['top_p']} K={cfg_d['top_k']} max={cfg_d['max_output_tokens']}")
    print(f"   Output tokens: {n_tok} | Response: {resp[:130]}{'...' if len(resp)>130 else ''}")
    print()


In [ ]:
# ── 5.4 Visualising Parameter Effects ────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# (A) Temperature on probability distribution
vocab = ["S3","EC2","Lambda","RDS","EKS","Glue","EMR","Redshift","DynamoDB","Kinesis"]
raw_logits = np.array([4.2, 3.8, 3.1, 2.7, 2.4, 2.0, 1.8, 1.5, 1.2, 0.9])

for ax, temp, col in zip(
    [axes[0], axes[0], axes[0], axes[0]],  # handled differently
    [0.1, 0.5, 1.0, 2.0], ['navy','steelblue','coral','firebrick']
):
    pass  # reset

ax0 = axes[0]
for temp, col, alpha in [(0.1,'navy',1.0),(0.5,'steelblue',0.8),(1.0,'coral',0.8),(2.0,'firebrick',0.7)]:
    probs = np.exp(raw_logits/temp); probs /= probs.sum()
    ax0.plot(vocab, probs, 'o-', color=col, linewidth=2, markersize=6, alpha=alpha, label=f"T={temp}")
ax0.set_title("Temperature Effect\non Token Probabilities", fontsize=11, fontweight='bold')
ax0.set_ylabel("Probability"); ax0.set_xticklabels(vocab, rotation=40, ha='right', fontsize=9)
ax0.legend(fontsize=9); ax0.grid(True, alpha=0.3)

# (B) Top-K filter
probs_std = np.exp(raw_logits/1.0); probs_std /= probs_std.sum()
colors_k = ['steelblue' if i < 5 else 'lightgrey' for i in range(10)]
axes[1].bar(vocab, probs_std, color=colors_k, edgecolor='navy', linewidth=0.8)
axes[1].axvline(4.5, color='red', linestyle='--', linewidth=2, label='Top-K=5 cutoff')
axes[1].set_title("Top-K Filtering\n(K=5, grey tokens excluded)", fontsize=11, fontweight='bold')
axes[1].set_ylabel("Probability"); axes[1].set_xticklabels(vocab, rotation=40, ha='right', fontsize=9)
axes[1].legend(fontsize=9); axes[1].grid(True, alpha=0.3, axis='y')

# (C) Top-P (nucleus) filter
cumulative = np.cumsum(np.sort(probs_std)[::-1])
p_cutoff   = 0.80
n_nucleus  = np.searchsorted(cumulative, p_cutoff) + 1
sorted_vocab = [v for _, v in sorted(zip(probs_std, vocab), reverse=True)]
colors_p = ['steelblue' if i < n_nucleus else 'lightgrey' for i in range(10)]
axes[2].bar(sorted_vocab, np.sort(probs_std)[::-1], color=colors_p, edgecolor='navy', linewidth=0.8)
axes[2].axvline(n_nucleus-0.5, color='red', linestyle='--', linewidth=2, label=f'Top-P=0.80 cutoff\n(nucleus={n_nucleus})')
axes[2].set_title("Top-P (Nucleus) Filtering\n(P=0.80, grey tokens excluded)", fontsize=11, fontweight='bold')
axes[2].set_ylabel("Probability"); axes[2].set_xticklabels(sorted_vocab, rotation=40, ha='right', fontsize=9)
axes[2].legend(fontsize=9); axes[2].grid(True, alpha=0.3, axis='y')

plt.suptitle("Module 5: Generation Parameter Effects on Token Sampling", fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig("/tmp/m5_params.png", dpi=130, bbox_inches='tight')
plt.show()


---
# MODULE 6 — Classical ML vs LLMs: Deep Comparison

| Dimension | Classical ML | LLMs |
|-----------|-------------|------|
| **Architecture** | Linear, Tree, SVM, shallow MLP | Deep Transformer (self-attention) |
| **Input format** | Fixed-size feature vector | Variable-length token sequence |
| **Training data** | Thousands–Millions labelled rows | Billions–Trillions tokens (mostly unlabelled) |
| **Training time** | Minutes–days | Weeks–months on thousands of GPUs |
| **Parameters** | 10s–Millions | Billions–Trillions |
| **Task setup** | Train one model per task | One model, many tasks via prompting |
| **Explainability** | High (SHAP, LIME, decision paths) | Low (attention maps only) |
| **Inference latency** | Microseconds–milliseconds | Milliseconds–seconds |
| **Labelled data** | Required | Minimal (few-shot or zero-shot) |
| **Output** | Scalar / class / probability | Any natural language or code |
| **AWS service** | SageMaker built-in algos, SKLearn | Bedrock, SageMaker JumpStart |


In [ ]:
# ── 6.1 Classical ML — TF-IDF + Random Forest Text Classifier ────────────
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report

# Labelled training data (REQUIRED for classical ML)
TRAIN_DATA = [
    ("EC2 instance not starting after AMI update",           "compute"),
    ("S3 bucket cross-account access denied",                "storage"),
    ("Lambda function timeout after 15 minutes",            "serverless"),
    ("RDS Aurora read replica replication lag",             "database"),
    ("EC2 CPU utilisation hitting 100 percent",              "compute"),
    ("S3 lifecycle policy not transitioning objects",        "storage"),
    ("Lambda cold start adding 4 second delay",             "serverless"),
    ("DynamoDB write capacity unit exceeded",               "database"),
    ("Auto Scaling not launching replacement instances",     "compute"),
    ("S3 multipart upload stuck incomplete",                 "storage"),
    ("API Gateway returning 502 from Lambda",               "serverless"),
    ("RDS PostgreSQL slow query performance",               "database"),
    ("EBS volume not visible after attach to EC2",          "compute"),
    ("S3 versioning deleting wrong object version",          "storage"),
    ("Lambda memory exhausted running ML inference",        "serverless"),
    ("Aurora MySQL backup window exceeding SLA",            "database"),
]

texts  = [t for t, _ in TRAIN_DATA]
labels = [l for _, l in TRAIN_DATA]

# Feature engineering
vectorizer = TfidfVectorizer(ngram_range=(1,2), min_df=1, max_features=500)
X = vectorizer.fit_transform(texts)

# Training
t0  = time.time()
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X, labels)
train_time_ms = (time.time() - t0) * 1000

print("CLASSICAL ML — Random Forest + TF-IDF")
print(f"Training samples     : {len(texts)}")
print(f"TF-IDF features      : {X.shape[1]}")
print(f"Training time        : {train_time_ms:.1f} ms")
print(f"Model parameters     : ~{clf.n_estimators * 50} (decision nodes)")
print(f"Labelled data needed : YES — {len(texts)} annotated examples\n")

# Inference test
test_tickets = [
    "EC2 instance terminated unexpectedly during peak traffic",
    "S3 objects not accessible from another AWS account",
    "Lambda function execution failing with out-of-memory error",
    "MySQL connection pool exhausted under load",
]

print("Inference results:")
for ticket in test_tickets:
    t0   = time.time()
    pred = clf.predict(vectorizer.transform([ticket]))[0]
    prob = max(clf.predict_proba(vectorizer.transform([ticket]))[0])
    ms   = (time.time() - t0) * 1000
    print(f"  Ticket: '{ticket[:55]}...'")
    print(f"  → Class: {pred:<12} | Confidence: {prob:.1%} | Latency: {ms:.2f}ms")


In [ ]:
# ── 6.2 LLM — Same Task, Zero Labelled Examples ───────────────────────────

model = genai.GenerativeModel(MODEL_NAME)
cfg0  = genai.GenerationConfig(max_output_tokens=20, temperature=0.0)

def llm_classify(ticket: str) -> tuple:
    prompt = (f"Classify this AWS ticket into exactly one of: compute | storage | serverless | database\n"
              f"Return ONLY the category word, lowercase.\n\nTicket: {ticket}")
    t0 = time.time()
    pred = model.generate_content(prompt, generation_config=cfg0).text.strip().lower()
    ms   = (time.time() - t0) * 1000
    return pred, ms

print("LLM ZERO-SHOT — No Training Data, No Feature Engineering")
print("Labelled data needed : NONE — uses pretrained world knowledge\n")
print("Inference results:")

for ticket in test_tickets:
    pred, ms = llm_classify(ticket)
    print(f"  Ticket: '{ticket[:55]}...'")
    print(f"  → Class: {pred:<12} | Latency: {ms:.0f}ms (includes network)")


In [ ]:
# ── 6.3 LLM Exclusive Capabilities ──────────────────────────────────────

model = genai.GenerativeModel(MODEL_NAME)
cfg   = genai.GenerationConfig(max_output_tokens=250, temperature=0.3)

tasks = [
    ("Multi-task (1 prompt, 3 outputs)",
     "Do ALL THREE tasks:\n1. Summarise in 1 sentence\n2. Extract action items as JSON list\n3. Classify urgency as LOW/MED/HIGH\n\nText: 'The SageMaker training endpoint in us-east-1 has been returning HTTP 503 since 14:30 UTC. The on-call team must check CloudWatch alarms, restart the endpoint, and notify the product team. Customer demo is in 3 hours.'"),
    
    ("Code Generation + Explanation",
     "Write a Python function using boto3 to list all SageMaker training jobs from the last 7 days, showing job name, status, and duration. Include docstring and error handling."),
    
    ("Complex Reasoning",
     "A model trained on US customer data is deployed in India and shows 23% accuracy drop. Give 4 specific technical reasons why, and suggest one AWS service that could help address each."),
]

print("LLM EXCLUSIVE CAPABILITIES — Classical ML Cannot Do These")

for task_name, prompt in tasks:
    answer = model.generate_content(prompt, generation_config=cfg).text.strip()
    print(f"\n{'='*60}")
    print(f"🎯 {task_name}")
    print(f"\n{answer}")


In [ ]:
# ── 6.4 Visual Comparison ────────────────────────────────────────────────

dimensions = [
    'Labelled\nData Need', 'Task\nFlexibility', 'Inference\nSpeed',
    'Explainability', 'Zero-shot\nAbility', 'Cost\nEfficiency',
    'Code/Text\nGeneration', 'Tabular\nData'
]
classical = [9, 2, 10, 9, 1, 9, 1, 9]
llm       = [2, 10, 4, 3, 10, 3, 10, 3]

x     = np.arange(len(dimensions))
width = 0.35

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Bar chart
b1 = ax1.bar(x - width/2, classical, width, label='Classical ML', color='2166AC', edgecolor='0D3D80', linewidth=1)
b2 = ax1.bar(x + width/2, llm,       width, label='LLM (Gemini Flash 2.5)', color='D6604D', edgecolor='A63020', linewidth=1)
for b in list(b1) + list(b2):
    ax1.text(b.get_x()+b.get_width()/2, b.get_height()+0.2, str(int(b.get_height())),
             ha='center', va='bottom', fontsize=9)
ax1.set_xticks(x); ax1.set_xticklabels(dimensions, fontsize=9)
ax1.set_ylim(0, 13); ax1.set_ylabel('Score (1=low, 10=high)', fontsize=11)
ax1.set_title('Classical ML vs LLM\nCapability Scores', fontsize=12, fontweight='bold')
ax1.legend(fontsize=10); ax1.grid(True, axis='y', alpha=0.3)

# Decision guide
scenarios = ['Tabular\nclassification', 'NLP / Text\ntasks', 'Real-time\nlow-latency', 
             'Few labelled\nexamples', 'Explainability\nrequired', 'Multi-task\none model']
rec_classical = [9, 2, 9, 2, 9, 1]
rec_llm       = [2, 10, 4, 9, 2, 10]

x2 = np.arange(len(scenarios))
ax2.barh(x2 + 0.2, rec_classical, 0.35, label='Classical ML wins', color='2166AC', alpha=0.85)
ax2.barh(x2 - 0.2, rec_llm,       0.35, label='LLM wins',          color='D6604D', alpha=0.85)
ax2.set_yticks(x2); ax2.set_yticklabels(scenarios, fontsize=10)
ax2.set_xlabel('Recommendation Score (1-10)', fontsize=11)
ax2.set_title('When to Choose Which?\nScenario Recommendations', fontsize=12, fontweight='bold')
ax2.legend(fontsize=10); ax2.grid(True, axis='x', alpha=0.3)
ax2.set_xlim(0, 13)

plt.suptitle('Module 6: Classical ML vs LLMs', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig("/tmp/m6_comparison.png", dpi=130, bbox_inches='tight')
plt.show()

print("\n📌 Rule of thumb:")
print("   Classical ML → structured/tabular data, latency-critical, explainability required")
print("   LLMs         → unstructured text, multi-task, few labels, complex reasoning needed")


---
# 🎓 Lab Complete — Summary

| Module | Concept | AWS Service |
|--------|---------|-------------|
| 1 | LLM Training & Artifacts | SageMaker Training + S3 checkpoints |
| 2 | Tool Calling | Lambda (tool executor) + API Gateway |
| 3 | Prompt Engineering | Bedrock system prompts + Guardrails |
| 4 | RAG Pipeline | OpenSearch Serverless + Bedrock KB |
| 5 | Generation Parameters | Bedrock inference config |
| 6 | Classical ML vs LLMs | SageMaker built-ins vs JumpStart |

## Next Steps
- Replace FAISS with **Amazon OpenSearch Serverless** k-NN for production RAG
- Deploy RAG as **Lambda + API Gateway** endpoint with streaming
- Add **Amazon Bedrock Guardrails** for content filtering and PII redaction
- Implement **RAGAS evaluation** (faithfulness, context recall, answer relevancy)
- Set up **CloudWatch** dashboards for LLM latency, token usage, cost monitoring


In [ ]:
print("""
╔══════════════════════════════════════════════════════════════╗
║     AWS LLM FOUNDATIONS LAB — COMPLETE ✅                    ║
║     BITS Pilani | Professional AI/ML Programme               ║
╠══════════════════════════════════════════════════════════════╣
║  ✅ Module 1: LLM Training & Artifacts (SageMaker + S3)      ║
║  ✅ Module 2: Tool Calling (Search + Docs + Code Interpreter) ║
║  ✅ Module 3: Prompt Engineering (System, CoT, Few-shot)      ║
║  ✅ Module 4: Full RAG Pipeline (FAISS + Gemini Flash 2.5)    ║
║  ✅ Module 5: Tokenisation + Temperature + Top-P/K            ║
║  ✅ Module 6: Classical ML vs LLMs (deep comparison)          ║
╚══════════════════════════════════════════════════════════════╝
""")
